# 01 – What is EDA?

**Exploratory Data Analysis (EDA)** is the first thing you do after loading a dataset — before any modelling, before any predictions, before any conclusions.

The idea is simple: *look at your data before you try to do anything with it.*

When you get a new dataset, you don't know:
- What's actually in it
- Whether the values are realistic
- Which columns are related to each other
- What patterns or surprises are hiding in there

EDA answers all of these questions. It's detective work — you're trying to understand what the data is saying before you ask it to answer a specific question.

## Why EDA Matters

Here's what happens when you skip EDA:

- You build a model on data that has 30% missing values — and your model learns mostly from the rows that happened to be filled in
- You calculate the average salary of a company without noticing one row says ₹50,00,000 per month — and your "average" is meaningless
- You build a classifier to detect fraud but never looked at the class balance — so your model predicts "not fraud" 100% of the time and is 99% accurate on paper

EDA catches these problems. It's the difference between working with data and *understanding* your data.

## The EDA Process

There is no single fixed workflow, but most EDA follows this general sequence:

```
1. Load and inspect the data
      ↓
2. Understand the shape and types
      ↓
3. Examine individual columns (univariate analysis)
      ↓
4. Look at pairs of columns together (bivariate analysis)
      ↓
5. Find and handle missing values and outliers
      ↓
6. Summarise what you found
```

You won't always do every step in exactly this order. Sometimes you find something unexpected in step 3 that sends you back to step 2. EDA is iterative — you go back and forth as you learn more.

## Setting Up — The Dataset We'll Use

We'll work with a student performance dataset throughout this section. It has realistic messy-data problems baked in — missing values, outliers, skewed distributions — so you get to deal with the same things you'd face on a real project.

In [ ]:
import pandas as pd
import numpy as np

# Reproducible random data
rng = np.random.default_rng(42)

n = 200

df = pd.DataFrame({
    "Student_ID": range(1001, 1001 + n),
    "Name":       rng.choice(["Amit","Priya","Ravi","Neha","Karan","Sneha",
                               "Rahul","Meera","Vijay","Pooja"], size=n),
    "Gender":     rng.choice(["Male","Female"], size=n, p=[0.52, 0.48]),
    "City":       rng.choice(["Mumbai","Pune","Delhi","Bangalore","Chennai"], size=n),
    "Department": rng.choice(["Science","Arts","Commerce"], size=n, p=[0.4,0.3,0.3]),
    "Study_Hours": np.clip(rng.normal(5, 2, n).round(1), 0, 12),
    "Attendance":  np.clip(rng.normal(75, 15, n).round(1), 20, 100),
    "Math":        np.clip(rng.normal(68, 18, n).round(0).astype(int), 0, 100),
    "Science":     np.clip(rng.normal(65, 20, n).round(0).astype(int), 0, 100),
    "English":     np.clip(rng.normal(70, 15, n).round(0).astype(int), 0, 100),
    "Fees_Paid":   rng.choice([True, False], size=n, p=[0.78, 0.22]),
})

# Inject some missing values (realistic)
for col, frac in [("Study_Hours", 0.05), ("Attendance", 0.03),
                  ("Math", 0.04), ("Science", 0.04), ("English", 0.03)]:
    idx = rng.choice(n, size=int(n * frac), replace=False)
    df.loc[idx, col] = np.nan

# Inject a few outliers
df.loc[rng.choice(n, 3, replace=False), "Math"]    = rng.integers(0, 15, 3)
df.loc[rng.choice(n, 2, replace=False), "Science"] = rng.integers(0, 10, 2)

# Add a Total column
df["Total"] = df[["Math","Science","English"]].sum(axis=1)

print(df.head(10))

This is our dataset — 200 students with marks, attendance, study hours, and demographic info.

Notice that `head()` only shows 10 rows. We have no idea yet whether the rest of the data looks like this. That's exactly why we need EDA — to understand the full picture, not just the first few rows.

## Step 1 — Shape and Basic Info

The first three questions you always answer:
1. How many rows and columns?
2. What are the data types?
3. Are there any missing values?

In [ ]:
# Shape — rows × columns
print("Shape:", df.shape)
print(f"We have {df.shape[0]} students and {df.shape[1]} columns.")

200 rows and 13 columns. For a student dataset, that's manageable — you can look at individual values if needed. For a million-row dataset, you'd rely entirely on summaries and charts.

In [ ]:
# Column names and data types
df.info()

`info()` tells you:
- Column name and position
- How many non-null values (missing = total rows minus this number)
- Data type — `int64`, `float64`, `object` (text), `bool`

Already useful: you can see which columns have fewer non-null values than the total 200 — those have missing data.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %":     missing_pct
})

print(missing_summary[missing_summary["Missing Count"] > 0])

No column has more than 5% missing values — that's minor. We'll fill them in later rather than drop rows.

This is the kind of decision you make *because* of EDA — you see that missing data is small and scattered, so dropping rows would be unnecessary data loss.

## Step 2 — Descriptive Statistics

`describe()` gives you the statistical summary of every numeric column in one shot.

In [ ]:
# Statistical summary — all numeric columns
print(df.describe().round(2))

Read this output column by column:

- **count** — how many non-null values (lower than 200 means there are NaNs)
- **mean** — average
- **std** — standard deviation (how spread out the values are)
- **min / max** — the extremes
- **25%, 50%, 75%** — quartiles: the value below which 25%, 50%, 75% of the data falls

Right away you can spot things:
- `Math` min is 0 and max is 100 — plausible range
- `Study_Hours` max is 12 — looks reasonable
- `Total` could go up to 300 (100 × 3 subjects)

The 50th percentile (median) is the middle value. If it's very different from the mean, your data has outliers or a skewed shape.

In [ ]:
# describe() for text/categorical columns
print(df.describe(include="object"))

For text columns, `describe()` shows:
- **count** — non-null values
- **unique** — how many distinct values
- **top** — the most frequent value
- **freq** — how many times that value appears

A quick sanity check: if `unique` equals `count` for a column that should have categories (like City), something went wrong in data creation.

## Step 3 — Value Counts for Categorical Columns

For text columns, you want to know how the values are distributed — not just which one appears most often.

In [ ]:
# Distribution across departments
print("Department distribution:")
print(df["Department"].value_counts())
print()

# As a percentage
print("As percentage:")
print((df["Department"].value_counts(normalize=True) * 100).round(1))

Science has about 40% of students — that's by design in our data. In a real dataset this might tell you something: one department is dominant, so overall statistics will be skewed by Science students.

In [ ]:
# Gender and city breakdown
print("Gender:")
print(df["Gender"].value_counts())
print()

print("City:")
print(df["City"].value_counts())

Even distribution of cities means no city dominates the analysis. Uneven distributions in a real dataset can be a sign of sampling bias — be careful when comparing groups of very different sizes.

## Step 4 — Spotting Outliers Quickly

Outliers are values that are far away from the rest of the data. They can be real (a student who genuinely scored 2 out of 100) or errors (a data entry mistake).

In [ ]:
# Who has very low Math scores?
low_math = df[df["Math"] < 15].dropna(subset=["Math"])
print(f"Students with Math < 15: {len(low_math)}")
print(low_math[["Student_ID", "Name", "Math", "Science", "English"]])

A few students with extremely low Math scores — these are the outliers we injected. In a real dataset, you'd investigate: did these students not turn up? Is this a data entry error? Are they real scores?

EDA doesn't automatically answer this. It just points you to where questions need to be asked.

In [ ]:
# Quick outlier check using the IQR method
# IQR = Q3 - Q1
# Outliers are typically below Q1 - 1.5×IQR or above Q3 + 1.5×IQR

def count_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), round(lower, 1), round(upper, 1)

for col in ["Math", "Science", "English", "Study_Hours", "Attendance"]:
    n_out, lo, hi = count_outliers(df[col].dropna())
    print(f"{col:<15}: {n_out} outliers  (expected range: {lo} to {hi})")

The IQR method gives you a mathematical definition of outliers. Values outside the `lower` to `upper` range are flagged.

This is just a starting point — whether to keep or remove outliers depends on context. A 0 in Math could be an absent student (valid) or a data entry error (invalid).

## Step 5 — Simple Relationships

Before making charts, you can spot relationships numerically. A common quick check: do students who study more score higher?

In [ ]:
# Correlation matrix — shows how strongly columns move together
# Values range from -1 (opposite) to +1 (same direction), 0 = no relationship

corr = df[["Study_Hours", "Attendance", "Math", "Science", "English", "Total"]].corr()
print(corr.round(2))

The correlation matrix shows every pair of columns. Key things to look for:
- High positive values (close to 1.0) — columns tend to go up together
- High negative values (close to -1.0) — one goes up as the other goes down
- Values near 0 — no linear relationship

In this dataset, `Math`, `Science`, and `English` should correlate positively with each other (students who are generally good tend to do well across subjects) — and you can see that in the numbers.

## Step 6 — Grouping by Category

A powerful EDA question: *Does this metric change depending on which group a student is in?*

In [ ]:
# Average marks by department
dept_summary = df.groupby("Department")[["Math","Science","English","Total"]].mean().round(1)
print(dept_summary)

Already interesting — Science students don't necessarily get the highest Science scores just because they're in that department. In real data, you often find that category labels and performance don't align the way you'd expect.

In [ ]:
# Average attendance and study hours by gender
print(df.groupby("Gender")[["Study_Hours","Attendance","Total"]].mean().round(1))

Group comparisons like this are where EDA starts generating real hypotheses — ideas worth investigating more carefully.

## What Comes Next

So far we've done EDA using only numbers — counts, means, correlations, value counts.

But data becomes much easier to understand when you *see* it. The patterns that take ten lines of numbers to describe often become obvious in a single chart.

The next five notebooks move from numbers into visuals:
- **Matplotlib basics** — how to build charts from scratch
- **Histograms and distributions** — understand the shape of your data
- **Scatter plots and correlation** — explore relationships between columns
- **Writing insights** — how to describe what you see

Throughout all of it, we'll keep coming back to this same student dataset.

## Summary

- **EDA** = exploring your data before doing anything else with it
- Always start with: shape → dtypes → missing values → describe()
- Use `value_counts()` for categorical columns
- Use the IQR method to flag outliers
- Use `corr()` to find linear relationships between numeric columns
- Use `groupby()` to compare groups
- EDA is iterative — you learn something, which raises a new question

Next up: **Matplotlib Basics** — the foundation for all data visualisation in Python.

## Practice Questions 1

1. Load the `df` dataset from this notebook and print the number of unique cities and departments.
2. Which city has the highest average `Total` score? Use `groupby()`.
3. How many students scored above 200 in `Total`? What percentage is that?

## Practice Questions 2

1. Check whether `Fees_Paid` is related to `Total` score — use `groupby("Fees_Paid")["Total"].mean()`.
2. Find the student with the highest `Total` score and the student with the lowest.
3. Calculate the correlation between `Study_Hours` and `Total`. What does the value tell you?